# 1.1 Validate Sparse Edges

Spot-check the computed edge table for correctness and coverage.

**Checks:**
1. Edge count distribution per school (are there outliers or suspiciously empty regions?)
2. Road-to-haversine ratio distribution (flag extreme detour factors)
3. Sea-separated edges in Visayas
4. Cross-region edge coverage at boundaries
5. Sample distance comparisons (manual check vs Google Maps)

In [1]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR))

from modules.coordinates import load_unified, list_regions
from modules.inter_island import tag_island_group
from modules.gcs_utils import OUTPUT_DIR

EDGES_DIR = OUTPUT_DIR / "edges"

# Load data
df_all = load_unified()
df_all = tag_island_group(df_all)
df_edges = pd.read_parquet(EDGES_DIR / "all_edges.parquet")

print(f"Schools: {len(df_all):,}")
print(f"Edges: {len(df_edges):,}")

  Excluded 770 private schools with suspect coordinates
Loaded 56,018 schools with valid coordinates
  Public:  47,874
  Private: 8,144
  Dropped 562 schools without coordinates
Schools: 56,018
Edges: 9,880,568


## 1. Edge count distribution per school

In [2]:
# Count edges per school (as source)
src_counts = df_edges.groupby("source_id").size().rename("n_edges")

# Merge with school data to get region
df_counts = df_all[["school_id", "region", "sector", "island_group"]].merge(
    src_counts, left_on="school_id", right_index=True, how="left"
)
df_counts["n_edges"] = df_counts["n_edges"].fillna(0).astype(int)

# Summary by region
region_summary = (
    df_counts.groupby("region")
    .agg(
        n_schools=("school_id", "count"),
        mean_edges=("n_edges", "mean"),
        median_edges=("n_edges", "median"),
        max_edges=("n_edges", "max"),
        pct_isolated=("n_edges", lambda x: (x == 0).mean() * 100),
    )
    .sort_values("mean_edges", ascending=False)
)
print("Edge counts by region:")
print(region_summary.round(1).to_string())

# Overall distribution
print(f"\nOverall edge count distribution:")
print(df_counts["n_edges"].describe().to_string())

# Top 10 most connected schools
top = df_counts.nlargest(10, "n_edges")[["school_id", "region", "sector", "n_edges"]]
print(f"\nTop 10 most connected schools:")
print(top.to_string(index=False))

# Isolated schools by region
isolated = df_counts[df_counts["n_edges"] == 0]
print(f"\nIsolated schools (0 edges): {len(isolated):,}")
if len(isolated) > 0:
    print(isolated.groupby("region").size().sort_values(ascending=False).head(10).to_string())

Edge counts by region:
             n_schools  mean_edges  median_edges  max_edges  pct_isolated
region                                                                   
NCR               1767      1349.1        1384.0       1869           0.1
Region IV-A       5383       380.9         276.0       1736           0.2
Region III        4747       271.6         237.0       1751           0.0
Region I          3270       180.8         145.0        476           0.3
Region VII        3274       161.8         110.0        538           0.2
Region VI         3257       134.3         115.0        419           0.2
BARMM             2580       111.3          97.0        273           2.0
Region V          4205        99.9          78.0        614           0.2
Region XI         2507        93.3          62.0        325           0.1
NIR               2565        88.3          71.0        297           0.1
Region X          2853        86.9          78.0        217           0.2
Region II      

## 2. Road-to-haversine ratio distribution

Ratio > 2.0 suggests winding mountain roads or river crossings.
Ratio < 1.0 is impossible (would mean road is shorter than straight line) — flag as data errors.

In [3]:
ratio = df_edges["road_haversine_ratio"].dropna()

print("Road-to-haversine ratio distribution:")
print(ratio.describe().to_string())

# Flag anomalies
below_one = (ratio < 1.0).sum()
above_three = (ratio > 3.0).sum()
above_five = (ratio > 5.0).sum()
print(f"\nAnomalies:")
print(f"  Ratio < 1.0 (impossible): {below_one:,}")
print(f"  Ratio > 3.0 (extreme detour): {above_three:,}")
print(f"  Ratio > 5.0 (very extreme): {above_five:,}")

# Percentile breakdown
percentiles = [50, 75, 90, 95, 99]
print(f"\nPercentiles:")
for p in percentiles:
    print(f"  p{p}: {ratio.quantile(p/100):.2f}")

# Highest ratio edges (inspect for plausibility)
if above_five > 0:
    extreme = df_edges.nlargest(10, "road_haversine_ratio")[
        ["source_id", "target_id", "road_distance_m", "haversine_distance_m", "road_haversine_ratio"]
    ].copy()
    extreme["road_km"] = extreme["road_distance_m"] / 1000
    extreme["hav_km"] = extreme["haversine_distance_m"] / 1000
    print(f"\nTop 10 extreme detour edges:")
    print(extreme[["source_id", "target_id", "road_km", "hav_km", "road_haversine_ratio"]].to_string(index=False))

Road-to-haversine ratio distribution:
count    9.880568e+06
mean     1.468575e+00
std      5.657139e-01
min      2.474613e-05
25%      1.245960e+00
50%      1.380434e+00
75%      1.571988e+00
max      8.023903e+02

Anomalies:
  Ratio < 1.0 (impossible): 137,152
  Ratio > 3.0 (extreme detour): 108,058
  Ratio > 5.0 (very extreme): 17,225

Percentiles:
  p50: 1.38
  p75: 1.57
  p90: 1.85
  p95: 2.12
  p99: 3.07

Top 10 extreme detour edges:
source_id target_id  road_km   hav_km  road_haversine_ratio
   406872   1500019   1.6252 0.002025            802.390320
   228002    304946   8.0951 0.016883            479.480225
   228002    320612   8.0951 0.016883            479.480225
   301444    109407   5.1610 0.052616             98.088173
   134264    134268   5.6600 0.058252             97.163971
   134268    134264   5.6600 0.058252             97.163971
   482940    409526   0.7765 0.009732             79.788513
   119898    303151   1.0719 0.015227             70.395782
   500529    5024

## 3. Sea-separated edges and island coverage

In [4]:
# Sea-separated edges
if "is_sea_separated" in df_edges.columns:
    sea = df_edges[df_edges["is_sea_separated"]]
    print(f"Sea-separated edges: {len(sea):,}")

    if len(sea) > 0:
        # Which regions have sea-separated edges?
        print(f"\nSea-separated by source region:")
        print(sea["source_region"].value_counts().to_string())

        # Sample
        print(f"\nSample sea-separated edges:")
        sample_cols = ["source_id", "target_id", "haversine_distance_m", "source_region"]
        print(sea[sample_cols].head(10).to_string(index=False))
else:
    print("No is_sea_separated column found.")

# Island group coverage
print(f"\nEdge count by island group:")
src_island = df_all.set_index("school_id")["island_group"]
df_edges_tmp = df_edges.copy()
df_edges_tmp["src_island"] = df_edges_tmp["source_id"].map(src_island)
df_edges_tmp["tgt_island"] = df_edges_tmp["target_id"].map(src_island)
print(df_edges_tmp.groupby("src_island").size().to_string())

# Cross-island edges (should be zero or very few — only via bridges/ferries in OSM)
cross_island = df_edges_tmp[df_edges_tmp["src_island"] != df_edges_tmp["tgt_island"]]
print(f"\nCross-island edges: {len(cross_island):,}")
if len(cross_island) > 0:
    print(cross_island[["source_id", "target_id", "src_island", "tgt_island", "road_distance_m"]].head(10).to_string(index=False))

Sea-separated edges: 0

Edge count by island group:
src_island
Luzon       7230776
Mindanao    1242849
Visayas     1406943

Cross-island edges: 0


## 4. Cross-region edge coverage

Verify that border municipalities have cross-region edges.

In [5]:
cross = df_edges[df_edges["is_cross_region"] == True]
print(f"Cross-region edges: {len(cross):,}")

if len(cross) > 0:
    # Which region pairs have cross-region edges?
    tgt_region = df_all.set_index("school_id")["region"]
    cross_tmp = cross.copy()
    cross_tmp["target_region"] = cross_tmp["target_id"].map(tgt_region)
    pair_counts = (
        cross_tmp.groupby(["source_region", "target_region"])
        .size()
        .sort_values(ascending=False)
    )
    print(f"\nCross-region edge pairs:")
    print(pair_counts.to_string())

    # Sample cross-region edges
    print(f"\nSample cross-region edges:")
    sample = cross_tmp.sample(min(10, len(cross_tmp)), random_state=42)
    print(sample[["source_id", "source_region", "target_id", "target_region", "road_distance_m"]].to_string(index=False))
else:
    print("No cross-region edges found. Check if cross_region_pairs.parquet was computed.")

Cross-region edges: 1,265,212

Cross-region edge pairs:
source_region  target_region
Region IV-A    NCR              444809
NCR            Region IV-A      434080
Region III     NCR              132295
NCR            Region III       128005
Region III     Region I          17223
Region I       Region III        17207
BARMM          Region XII        15482
Region XII     BARMM             15473
CAR            Region II          6874
Region II      CAR                6865
Region IX      Region X           5347
Region X       Region IX          5340
Region IV-A    Region III         4577
Region III     Region IV-A        4071
Region I       CAR                3088
CAR            Region I           3082
Region XI      Region XII         2624
Region XII     Region XI          2621
BARMM          Region IX          1508
Region IX      BARMM              1506
Region V       Region IV-A        1169
Region IV-A    Region V           1127
Region III     Region II           901
Region II      Reg

## 5. Sample distance verification

Pick random school pairs and print their OSRM distances alongside coordinates
so you can manually verify against Google Maps.

**How to verify:** Copy the coordinates below into Google Maps directions and
compare the driving distance.

In [6]:
# Sample 20 edges across different distance ranges
np.random.seed(42)

coord_lookup = df_all.set_index("school_id")[["school_name", "latitude", "longitude", "region"]]

# Stratify: 5 short (<5km), 5 medium (5-10km), 5 long (10-15km), 5 very long (15-20km)
bins = [(0, 5000), (5000, 10000), (10000, 15000), (15000, 20000)]
samples = []

for lo, hi in bins:
    band = df_edges[(df_edges["road_distance_m"] >= lo) & (df_edges["road_distance_m"] < hi)]
    if len(band) > 0:
        n = min(5, len(band))
        samples.append(band.sample(n, random_state=42))

df_sample = pd.concat(samples, ignore_index=True)

print(f"Verification sample: {len(df_sample)} edges\n")
print(f"{'Source ID':<12} {'Target ID':<12} {'Road km':>8} {'Hav km':>8} {'Ratio':>6} {'Source coords':<25} {'Target coords':<25} {'Region'}")
print("-" * 120)

for _, row in df_sample.iterrows():
    src = coord_lookup.loc[row["source_id"]] if row["source_id"] in coord_lookup.index else None
    tgt = coord_lookup.loc[row["target_id"]] if row["target_id"] in coord_lookup.index else None

    road_km = row["road_distance_m"] / 1000
    hav_km = row["haversine_distance_m"] / 1000
    ratio = row["road_haversine_ratio"]

    src_coords = f"{src['latitude']:.5f},{src['longitude']:.5f}" if src is not None else "N/A"
    tgt_coords = f"{tgt['latitude']:.5f},{tgt['longitude']:.5f}" if tgt is not None else "N/A"
    region = src["region"] if src is not None else "N/A"

    print(f"{row['source_id']:<12} {row['target_id']:<12} {road_km:>8.2f} {hav_km:>8.2f} {ratio:>6.2f} {src_coords:<25} {tgt_coords:<25} {region}")

Verification sample: 20 edges

Source ID    Target ID     Road km   Hav km  Ratio Source coords             Target coords             Region
------------------------------------------------------------------------------------------------------------------------
100040       400002           3.01     2.51   1.20 17.90694,120.46710        17.92755,120.47674        Region I
403106       109485           1.60     1.47   1.09 14.68764,121.11692        14.67636,121.10973        Region IV-A
406791       319901           2.78     1.90   1.46 14.65400,121.09810        14.67065,121.10207        NCR
122541       122543           2.13     1.52   1.40 11.04381,125.72497        11.05686,125.72086        Region VIII
106564       106577           3.27     3.15   1.04 15.67420,120.58474        15.64611,120.58890        Region III
401810       406958           6.10     4.32   1.41 14.57920,121.06494        14.59858,121.03012        Region IV-A
107054       106080           6.83     5.22   1.31 15.12893,

## 6. Per-region file sizes

Check that all regions produced output and note file sizes.

In [6]:
print(f"{'File':<40} {'Size (MB)':>10} {'Rows':>12}")
print("-" * 65)

total_size = 0
total_rows = 0
for f in sorted(EDGES_DIR.glob("*.parquet")):
    size_mb = f.stat().st_size / 1e6
    rows = len(pd.read_parquet(f, columns=["source_id"]))
    total_size += size_mb
    total_rows += rows
    print(f"{f.name:<40} {size_mb:>10.2f} {rows:>12,}")

print("-" * 65)
print(f"{'TOTAL':<40} {total_size:>10.2f} {total_rows:>12,}")

# Check manifest
manifest_path = EDGES_DIR / "_manifest.json"
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    print(f"\nManifest created: {manifest['created_at']}")
    print(f"Parameters: cutoff={manifest['parameters']['road_cutoff_m']/1000:.0f}km, "
          f"profile={manifest['parameters']['osrm_profile']}")
else:
    print("\nWARNING: No manifest found. Run Section 6.0 of notebook 1.0.")

File                                      Size (MB)         Rows
-----------------------------------------------------------------
all_edges.parquet                            128.29    9,880,568
cross_region_pairs.parquet                    16.97    1,265,212
region_barmm.parquet                           3.44      270,141
region_car.parquet                             1.76      123,734
region_caraga.parquet                          1.99      140,914
region_mimaropa.parquet                        1.58      118,614
region_ncr.parquet                            24.58    1,821,795
region_nir.parquet                             3.09      225,305
region_region_i.parquet                        7.55      570,647
region_region_ii.parquet                       3.13      234,978
region_region_iii.parquet                     15.01    1,134,691
region_region_iv_a.parquet                    21.56    1,599,098
region_region_ix.parquet                       3.07      225,488
region_region_v.parquet 

ArrowInvalid: No match for FieldRef.Name(source_id) in school_id: string
school_name: string
latitude: double
longitude: double
region: string
province: string
municipality: string
barangay: string
offers_es: string
offers_jhs: string
offers_shs: string
urban_rural: string
enrollment_status: string
psgc_region: string
psgc_province: string
psgc_municity: string
sector: string
esc_participating: double
shsvp_participating: double
jdvp_participating: double
island_group: string
osrm_status: string
__fragment_index: int32
__batch_index: int32
__last_in_fragment: bool
__filename: string